**N.B.** The "Simpson model" referred to here is actually the Poisson distribution, and the way in which the problem specifies how we need to code the cumulative distribution function is non-standard in a statistical sense, and in terms of convention, e.g. compared to software packages! Take care with the fact that `prob_simpson(8, 12, ">=")` refers to $P(c \leq 12)$ under the rubric of the kata, not $P(c \geq 12)$!!!

A chain of car rental locals made a statistical research for each local in a city. They measure the average of clients for every day.

They used the Simpson model for probabilities 

$$P(x) = \frac{\lambda^x e^{- \lambda}}{x!}$$

where:

* $\lambda$ -  average for the amount of events having the same conditions during the measurement.
* $x$ - number of the events different from the average at the same place and time interval for obtaining the average.

Suposse that one of the locals has an average 8 clients per day.

The probability of having 12 clients, one of the goals of the manager, will be:

$$P(\text{clients} = x) = \frac{\lambda^x e^{- \lambda}}{x!} = \frac{8^{12}e^{-8}}{12!} = 0.048126804281$$

It is a very low value, almost 5%. It shows that they have to work a lot for this increment.

And, how would it be if we want to know the sum of probabilities for values below 12 (cumulative probability)?

```
P(c < 12) = P(c=0) + P(c=1) + P(c=3) + P(c=4) + P(c=5) + P(c=6) + P(c=7) + P(c=8) + P(c=9) +  P(c=10) + P(c=11) 

P(c<12) = 0.00033546262790251196 + 0.0026837010232200957 + 0.010734804092880383 + 0.02862614424768102 + 0.05725228849536204 + 0.09160366159257927 + 0.12213821545677235 + 0.13958653195059698 + 0.13958653195059698 + 0.12407691728941954 + 0.09926153383153563 + 0.072190206422935 = 0.888075998981
```

But the probability of having below or equal the value 12, will be:

```
P(c≤12) = P(c=0) + P(c=1) + P(c=3) + P(c=4) + P(c=5) + P(c=6) + P(c=7) + P(c=8) + P(c=9) +  P(c=10) + P(c=11) + P(c=12) = 0.936202803263
(P(c≤12) = P(12≥c))
```

The probability for having equal or more clients than 12 will be:

```
P(c>12) = P(c=13) + P(c=14) + P(c=15) + ........+ P(c=+∞) 
(P(c>12) = P(12<c))
```

But the sum of the probabilities for c = 0 to +∞ is 1

So,

```
P(c>12) = 1 - P(c≤12) = 1 - 0.936202803263 = 0.063797196737
```

Make the function prob_simpson(), that will receive 2 or 3 arguments.

Two arguments if we want to calculate the probability for a specific number, receives the average lamb and the variable x.

Three arguments if we want the cumulative probability for a value, besides receiving lamb and x, will receive one of the following operators in string format: <, <=, >, >=.

Let's see some cases for probability at the number:

prob_simpson(8, 12) == 0.04812680428195667
prob_simpson(8, 11) == 0.072190206422935
prob_simpson(8, 10) == 0.09926153383153563
And for cumulative probability:

prob_simpson(8, 12, '>') == 0.888075998981    # or P(12>c)
prob_simpson(8, 12, '>=') ==  0.936202803263  # or P(12≥c)
prob_simpson(8, 12, '<') == 0.063797196737    # or P(12<c)
prob_simpson(8, 12, '<=') == 0.111924001019   # or P(12≤c)
You may assume that you will not receive high values for lamb and x This kata will also have translations into Javascript and Ruby.

Enjoy it!

AlgorithmsMathematicsStatisticsProbabilityData Science

* Ideas.
* We need to compute a formula for the probability mass function (PMF) and cumulative distribution function (CDF) of the Simpson distribution.
* Start with PMF first, as that is used to code the CDF.
* We often code the CDF using complementary events - script will need to parse the type of cumulative probability we will need and then work from there.
* Would it better to write the PMF as a function, then call that repeatedly within the CDF? Because we may be able to cache the computation of the factorial? Get code working then optimise if necessary.
* Also: Take care with the fact that `prob_simpson(8, 12, ">=")` refers to $P(c \leq 12)$, not $P(c \geq 12)$!!!

In [55]:
# Final submission.

import math

def factorial(n):
    if n == 0:
        return 1
    else:
        n_fact = 1
        for num in range(1, n + 1):
            n_fact *= num
        return n_fact
    
def prob_simpson(lamb, x, op="="):
    if op == "=":
        return (lamb ** x) * (math.e ** -lamb) / factorial(x)
    probs = [(lamb ** i) * (math.e ** -lamb) / factorial(i) for i in range(0, x + 1)]

    if op == ">=":
        return sum(probs) 
    elif op == ">":
        return sum(probs[0: x])
    elif op == "<=":
        return 1 - sum(probs[0 : x])
    elif op == "<":
        return 1 - sum(probs)

    return 0

In [56]:
prob_simpson(8, 12)

0.04812680428195667

* Code Review.
* `math` does not just contain $e$ as a constant, it also has a `factorial` function. Given that you've imported `math`, you may as well use it for numerical stability etc.
* We can define helper functions like your `factorial` function at the module level, outside `prob_simpson`, or as a nested function within `probs_simpson`. Former ensures that `factorial` isn't repeatedly defined anew everytime `prob_simpson` is called, and is also accessible by other functions, if necessary. The latter approach, encapsulation, means that is hidden from the global namespace, preventing name conflicts.
* Given that needed to access slice of `probs` for the cumulative probability calculations, using a list comprehension instead of a generator was a good choice.      

***

In [57]:
# Code the PMF.
import math

lamb = 8
x = 12

def factorial(n):
    if n == 0:
        return 1
    else:
        n_fact = 1
        for num in range(1, n + 1):
            n_fact *= num
        return n_fact

prob = (lamb ** x) * (math.e ** -lamb) / factorial(x)

In [58]:
# Code the CDF.

op = "<="

probs = [(lamb ** i) * (math.e ** -lamb) / factorial(i) for i in range(0, x + 1)]

if op == ">=":
    cumul_prob = sum(probs) 
elif op == ">":
    cumul_prob = sum(probs[0: x])
elif op == "<=":
    cumul_prob = 1 - sum(probs[0 : x])
elif op == "<":
    cumul_prob = 1 - sum(probs)